In [0]:
%run ./16-streaming-aggregates

In [0]:
class aggregationTestSuite():
    def __init__(self):
        import os
        self.base_data_dir = os.path.abspath("..")
        self.checkpoint_dir = ("/Volumes/spark_stream/default/streaming_data/checkpoints")

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists spark_stream.default.invoices_bz")
        spark.sql("drop table if exists spark_stream.default.customer_rewards")

        dbutils.fs.rm(f"{self.checkpoint_dir}/invoices_bz",True)
        dbutils.fs.rm(f"{self.checkpoint_dir}/customer_rewards",True)

        dbutils.fs.rm(f"{self.base_data_dir}/data/invoices", True)
        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/invoices")
        print("Done")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/invoices/invoices-{itr}.json", f"{self.base_data_dir}/data/invoices/")
        print("Done")

    def assertBronze(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql("select count(*) from spark_stream.default.invoices_bz").collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")  

    def assertGold(self, expected_value):
        print(f"\tStarting validation...", end='')
        actual_value = spark.sql('''select TotalAmount 
                                 from spark_stream.default.customer_rewards 
                                 where CustomerCardNo = '2262471989'
                                 ''').collect()[0][0]
        assert expected_value == actual_value, f"Test failed! actual count is {actual_value}"
        print("Done")  
  
    def runTests(self):
        self.cleanTests()

        bzStream = Bronze()
        gdStream = Gold()

        print("\nTesting first iteration of invoice stream...")
        self.ingestData(1)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()
        self.assertBronze(501)

        gdQuery = gdStream.process()
        gdQuery.awaitTermination()
        self.assertGold(36859)

        print("Validation passed.\n")

        print("Testing second iteration of invoice stream...")
        self.ingestData(2)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()
        self.assertBronze(501+500)

        gdQuery = gdStream.process()
        gdQuery.awaitTermination()
        self.assertGold(36859+20740)

        print("Validation passed.\n")

        print("Testing third iteration of invoice stream...")
        self.ingestData(3)

        bzQuery = bzStream.process()
        bzQuery.awaitTermination()
        self.assertBronze(501+500+590)

        gdQuery = gdStream.process()
        gdQuery.awaitTermination()
        self.assertGold(36859+20740+31959)

        print("Validation passed.\n")

In [0]:
aTS = aggregationTestSuite()
aTS.runTests()